In [1]:
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
from torch.utils.data import DataLoader, Dataset
from transformers import (
    LongformerTokenizer, LongformerForSequenceClassification,
    BertTokenizer, BertForSequenceClassification,
    AutoTokenizer, AutoModelForSequenceClassification
)
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report, precision_score, recall_score

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# ----- Longformer Dataset and Prediction -----
class LongformerTextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=4096, use_global_attention=True, global_attention_target='cls', dynamic_padding=False):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.global_attention_target = global_attention_target
        self.dynamic_padding = dynamic_padding
        self.use_global_attention = use_global_attention

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = self.labels[idx]
        padding_strategy = "longest" if self.dynamic_padding else "max_length"
        inputs = self.tokenizer(text, return_tensors="pt", truncation=True, padding=padding_strategy, max_length=self.max_length)
        input_ids = inputs["input_ids"].squeeze(0)
        attention_mask = inputs["attention_mask"].squeeze(0)
        output = {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'labels': torch.tensor(label, dtype=torch.float)
        }
        if self.use_global_attention:
            global_attention_mask = torch.zeros_like(input_ids)
            if self.global_attention_target == 'cls':
                global_attention_mask[0] = 1
            output['global_attention_mask'] = global_attention_mask
        return output

def get_longformer_predictions(model, data_loader, device):
    model.eval()
    predictions, true_labels, all_probs = [], [], []
    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            global_attention_mask = batch.get('global_attention_mask', None)
            if global_attention_mask is not None:
                global_attention_mask = global_attention_mask.to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, global_attention_mask=global_attention_mask)
            probabilities = torch.sigmoid(outputs.logits)
            predicted = (probabilities >= 0.5).float()
            predictions.extend(predicted.view(-1).cpu().numpy())
            true_labels.extend(labels.view(-1).cpu().numpy())
            all_probs.extend(probabilities.view(-1).cpu().numpy())
    return np.array(predictions), np.array(true_labels), np.array(all_probs)

In [4]:
# ==== BERT/CLINICALBERT DATASET (CHUNKED) ====
class ChunkedTextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, chunk_size=510, max_length=512, doc_max_length=4096):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.chunk_size = chunk_size
        self.max_length = max_length
        self.doc_max_length = doc_max_length

    def __len__(self):
        return len(self.texts)

    def chunk_text(self, text):
        tokens = self.tokenizer.encode(text, add_special_tokens=False)
        tokens = tokens[:self.doc_max_length]
        chunks = []
        for i in range(0, len(tokens), self.chunk_size):
            chunk = tokens[i:i+self.chunk_size]
            chunk = [self.tokenizer.cls_token_id] + chunk + [self.tokenizer.sep_token_id]
            if len(chunk) < self.max_length:
                chunk += [self.tokenizer.pad_token_id] * (self.max_length - len(chunk))
            chunk = chunk[:self.max_length]
            chunks.append(chunk)
        return chunks

    def __getitem__(self, idx):
        text = self.texts[idx]
        label = float(self.labels[idx])
        chunks = self.chunk_text(text)
        return {
            'chunks': torch.tensor(chunks, dtype=torch.long),
            'label': torch.tensor(label, dtype=torch.float),   # <-- make sure key is 'label'
            'num_chunks': len(chunks)
        }

def bert_collate_fn(batch):
    all_chunks = [item['chunks'] for item in batch]
    all_labels = torch.tensor([item['label'] for item in batch], dtype=torch.float)
    all_num_chunks = [item['num_chunks'] for item in batch]
    flat_chunks = torch.cat(all_chunks, dim=0)
    return {
        'chunks': flat_chunks,         # [total_chunks, max_length]
        'labels': all_labels,          # [batch_size]
        'num_chunks': all_num_chunks   # list of ints
    }

def get_bert_predictions(model, data_loader, device, tokenizer, pooling='max'):
    model.eval()
    all_preds, all_labels, all_probs = [], [], []
    with torch.no_grad():
        for batch in tqdm(data_loader, desc="Predicting"):
            chunks = batch['chunks'].to(device)
            labels = batch['labels'].to(device)
            num_chunks = batch['num_chunks']
            outputs = model(input_ids=chunks, attention_mask=(chunks != tokenizer.pad_token_id))
            logits = outputs.logits.view(-1)
            chunk_idx = 0
            pooled_probs = []
            for nc in num_chunks:
                chunk_logits = logits[chunk_idx:chunk_idx+nc]
                prob = torch.sigmoid(chunk_logits)
                if pooling == 'max':
                    pooled_prob = torch.max(prob)
                elif pooling == 'mean':
                    pooled_prob = torch.mean(prob)
                else:
                    raise ValueError(f"Unknown pooling: {pooling}")
                pooled_probs.append(pooled_prob)
                chunk_idx += nc
            pooled_probs = torch.stack(pooled_probs)
            preds = (pooled_probs >= 0.5).long()
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(pooled_probs.cpu().numpy())
    return np.array(all_preds), np.array(all_labels), np.array(all_probs)

In [5]:
# -------- Model Configurations --------
model_configs = [
    # Longformer
    {
        "name": "Longformer_MIMIC",
        "model_dir": "/content/drive/My Drive/EHR_PROJ/MODELS/mimic_no_preprocess_no_glob_binary",
        "tokenizer_cls": LongformerTokenizer,
        "model_cls": LongformerForSequenceClassification,
        "type": "longformer",
        "max_length": 4096
    },
    {
        "name": "Longformer_Berkeley_MIMIC",
        "model_dir": "/content/drive/My Drive/EHR_PROJ/MODELS/new_berkeley_pretrain_to_mimic_v010725",
        "tokenizer_cls": LongformerTokenizer,
        "model_cls": LongformerForSequenceClassification,
        "type": "longformer",
        "max_length": 4096
    },
    {
        "name": "Longformer_Berkeley_Phenotype_MIMIC",
        "model_dir": "/content/drive/My Drive/EHR_PROJ/MODELS/new_berkeley_pretrain_to_phenotype_to_mimic_v030325",
        "tokenizer_cls": LongformerTokenizer,
        "model_cls": LongformerForSequenceClassification,
        "type": "longformer",
        "max_length": 4096
    },
    # BERT
    {
        "name": "BERT_Hate_MIMIC",
        "model_dir": "/content/drive/My Drive/EHR_PROJ/MODELS/bert_hate_mimic_0522",
        "tokenizer_cls": BertTokenizer,
        "model_cls": BertForSequenceClassification,
        "type": "bert",
        "max_length": 512
    },
    {
        "name": "BERT_Hate_Phenotype_MIMIC",
        "model_dir": "/content/drive/My Drive/EHR_PROJ/MODELS/bert_hate_phenotype_mimic_0523",
        "tokenizer_cls": BertTokenizer,
        "model_cls": BertForSequenceClassification,
        "type": "bert",
        "max_length": 512
    },
    {
        "name": "BERT_MIMIC",
        "model_dir": "/content/drive/My Drive/EHR_PROJ/MODELS/mimic_bert_0522",
        "tokenizer_cls": BertTokenizer,
        "model_cls": BertForSequenceClassification,
        "type": "bert",
        "max_length": 512
    },
    # ClinicalBERT
    {
        "name": "ClinicalBERT_Hate_MIMIC",
        "model_dir": "/content/drive/My Drive/EHR_PROJ/MODELS/ClinicalBERT_hate_mimic_0523",
        "tokenizer_cls": AutoTokenizer,
        "model_cls": AutoModelForSequenceClassification,
        "type": "bert",
        "max_length": 512
    },
    {
        "name": "ClinicalBERT_Hate_Phenotype_MIMIC",
        "model_dir": "/content/drive/My Drive/EHR_PROJ/MODELS/ClinicalBERT_hate_phenotype_mimic_0524",
        "tokenizer_cls": AutoTokenizer,
        "model_cls": AutoModelForSequenceClassification,
        "type": "bert",
        "max_length": 512
    },
    {
        "name": "ClinicalBERT_MIMIC",
        "model_dir": "/content/drive/My Drive/EHR_PROJ/MODELS/clinicalbert_mimic_0523",
        "tokenizer_cls": AutoTokenizer,
        "model_cls": AutoModelForSequenceClassification,
        "type": "bert",
        "max_length": 512
    },
]

In [6]:
mimic_test = pd.read_csv('/content/drive/My Drive/EHR_PROJ/DATA/mimic_no_preprocess_binary_val.csv')
test_texts = mimic_test['text'].tolist()
test_labels = mimic_test['label'].tolist()

In [ ]:
NUM_SUBSETS = 35
SUBSET_FRAC = 0.7   # or 0.7 if you prefer 70% subsets
BATCH_SIZE = 32      # for chunked eval, keep at 1 unless you change code

device = 'cuda' if torch.cuda.is_available() else 'cpu'
all_subset_results = []

for subset_idx in range(NUM_SUBSETS):
    subset_indices = np.random.choice(len(test_texts), size=int(SUBSET_FRAC * len(test_texts)), replace=False)
    subset_texts = [test_texts[i] for i in subset_indices]
    subset_labels = [test_labels[i] for i in subset_indices]

    for cfg in model_configs:
        print(f"Subset {subset_idx+1}/{NUM_SUBSETS} -- Model: {cfg['name']}")
        tokenizer = cfg['tokenizer_cls'].from_pretrained(cfg['model_dir'])
        model = cfg['model_cls'].from_pretrained(cfg['model_dir'])
        model.to(device)

        # Create the right Dataset/DataLoader for each model
        if cfg.get('type') == 'longformer':
            subset_dataset = LongformerTextDataset(
                subset_texts, subset_labels, tokenizer,
                max_length=cfg['max_length'],
                use_global_attention=True,
                global_attention_target='cls',
                dynamic_padding=False
            )
            subset_loader = DataLoader(subset_dataset, batch_size=BATCH_SIZE, shuffle=False)
            preds, trues, probs = get_longformer_predictions(model, subset_loader, device)
        else:
            subset_dataset = ChunkedTextDataset(
                subset_texts, subset_labels, tokenizer,
                chunk_size=510, max_length=cfg['max_length'], doc_max_length=4096
            )
            subset_loader = DataLoader(
                subset_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=bert_collate_fn
            )
            preds, trues, probs = get_bert_predictions(model, subset_loader, device, tokenizer, pooling='max')

        # Compute metrics
        accuracy = accuracy_score(trues, preds)
        f1 = f1_score(trues, preds)
        precision = precision_score(trues, preds)
        recall = recall_score(trues, preds)
        try:
            auc = roc_auc_score(trues, probs)
        except Exception:
            auc = np.nan

        all_subset_results.append({
            "subset": subset_idx,
            "model": cfg['name'],
            "accuracy": accuracy,
            "f1": f1,
            "precision": precision,
            "recall": recall,
            "auc": auc
        })

Subset 1/35 -- Model: Longformer_MIMIC
Subset 1/35 -- Model: Longformer_Berkeley_MIMIC
Subset 1/35 -- Model: Longformer_Berkeley_Phenotype_MIMIC
Subset 1/35 -- Model: BERT_Hate_MIMIC


Predicting: 100%|██████████| 19/19 [00:43<00:00,  2.29s/it]


Subset 1/35 -- Model: BERT_Hate_Phenotype_MIMIC


Predicting: 100%|██████████| 19/19 [00:43<00:00,  2.29s/it]


Subset 1/35 -- Model: BERT_MIMIC


...

Predicting: 100%|██████████| 19/19 [00:27<00:00,  1.43s/it]


Subset 35/35 -- Model: ClinicalBERT_Hate_Phenotype_MIMIC


Predicting: 100%|██████████| 19/19 [00:27<00:00,  1.43s/it]


Subset 35/35 -- Model: ClinicalBERT_MIMIC


Predicting: 100%|██████████| 19/19 [00:27<00:00,  1.43s/it]


In [ ]:
subset_results_df = pd.DataFrame(all_subset_results)
subset_results_df.to_csv("/content/drive/My Drive/EHR_PROJ/Results/transformer_model_subset_eval_results.csv", index=False)

In [ ]:
print("\nAverage performance per model over all subsets:")
print(subset_results_df.groupby("model").mean(numeric_only=True)[["accuracy", "f1", "precision", "recall", "auc"]])


Average performance per model over all subsets:
                                     accuracy        f1  precision    recall  \
model                                                                          
BERT_Hate_MIMIC                      0.807514  0.871683   0.846708  0.898233   
BERT_Hate_Phenotype_MIMIC            0.837766  0.891259   0.870295  0.913326   
BERT_MIMIC                           0.759071  0.823571   0.882011  0.772468   
ClinicalBERT_Hate_MIMIC              0.813791  0.871875   0.873557  0.870304   
ClinicalBERT_Hate_Phenotype_MIMIC    0.835245  0.890050   0.865508  0.916116   
ClinicalBERT_MIMIC                   0.800593  0.867933   0.838012  0.900142   
Longformer_Berkeley_MIMIC            0.877558  0.917903   0.896676  0.940229   
Longformer_Berkeley_Phenotype_MIMIC  0.898270  0.931842   0.909401  0.955472   
Longformer_MIMIC                     0.860603  0.905608   0.892969  0.918674   

                                          auc  
model                 